# League of Legends: Forward Propagation

We use pretend LoL stats to learn how a network makes predictions. The scores are only examples, not real match predictions.


### What is a Neural Network?

In [1]:
# LoL example (SISO = one input, one output):
# Input = our champion's CS per minute (8.5).
# Weight = how strongly CS affects our pretend performance score (0.1).
# Multiply them to get one prediction: 8.5 * 0.1 = 0.85.
# Bigger input or weight means a bigger score here.

# The network:

weight = 0.1 
def neural_network(input, weight):
    prediction = input * weight
    return prediction

# How we use the network to predict something:

cs_per_min = [8.5, 9.5, 10, 9]
input = cs_per_min[0]
pred = neural_network(input,weight)
print(pred)

0.8500000000000001


# Making a Prediction with Multiple Inputs

### Complete Runnable Code

In [2]:
# LoL example (MISO = many inputs, one output):
# We use CS/min, recent win rate, and a scaled vision score.
# w_sum starts at 0, multiplies each stat by its weight, then adds them.
# Step 1: 8.5 * 0.1 = 0.85 (CS).
# Step 2: 0.65 * 0.2 = 0.13 (win rate).
# Step 3: 1.2 * 0 = 0 (vision). Total = 0.98.
# Vision's weight is 0, so this toy model ignores that stat.

def w_sum(a,b):
    assert(len(a) == len(b))
    output = 0
    for i in range(len(a)):
        output += (a[i] * b[i])
    return output

weights = [0.1, 0.2, 0] 
    
def neural_network(input, weights):
    pred = w_sum(input,weights)
    return pred

# This dataset is the current
# status at the beginning of
# each game for four League of Legends matches.

# cs_per_min = CS per minute
# win_rate = recent win rate
# vision_score = vision score (scaled)

cs_per_min =  [8.5, 9.5, 9.9, 9.0]
win_rate = [0.65, 0.8, 0.8, 0.9]
vision_score = [1.2, 1.3, 0.5, 1.0]

# Input corresponds to every entry
# for the first game of the season.

input = [cs_per_min[0],win_rate[0],vision_score[0]]
pred = neural_network(input,weights)

print(pred)

0.9800000000000001


### NumPy Code

In [3]:
# Same LoL match, but NumPy does the work.
# input.dot(weights) multiplies matching entries and adds them.
# So it gives 0.85 + 0.13 + 0 = 0.98, just like w_sum.
# Both arrays have 3 values, so their sizes match.

import numpy as np
weights = np.array([0.1, 0.2, 0])
def neural_network(input, weights):
    pred = input.dot(weights)
    return pred
    
cs_per_min =  np.array([8.5, 9.5, 9.9, 9.0])
win_rate = np.array([0.65, 0.8, 0.8, 0.9])
vision_score = np.array([1.2, 1.3, 0.5, 1.0])

# Input corresponds to every entry
# for the first game of the season.

input = np.array([cs_per_min[0],win_rate[0],vision_score[0]])
pred = neural_network(input,weights)

print(pred)

0.9800000000000001


# Making a Prediction with Multiple Outputs

In [4]:
# LoL example (SIMO = one input, many outputs):
# One input (win rate = 0.65) makes three pretend scores.
# Each output uses its own weight: 0.65 * 0.3, * 0.2, * 0.9.
# Before, many stats fed one result. Now one stat feeds three results.

# One recent win rate can give three pretend outputs:
# damage taken, chance to win, and tilt level.
# These are toy scores, not real game statistics.

def ele_mul(number,vector):
    output = [0,0,0]
    assert(len(output) == len(vector))
    for i in range(len(vector)):
        output[i] = number * vector[i]
    return output

weights = [0.3, 0.2, 0.9] 

def neural_network(input, weights):
    pred = ele_mul(input,weights)
    return pred
    
win_rate = [0.65, 0.8, 0.8, 0.9]
input = win_rate[0]
pred = neural_network(input,weights)

print(pred)

[0.195, 0.13, 0.5850000000000001]


# Predicting with Multiple Inputs & Outputs

In [5]:
# LoL example (MIMO = many inputs, many outputs):
# Columns = CS/min, win rate, vision score.
# Rows = damage taken score, win chance score, tilt score.
# Each row has the 3 weights for ONE output.
# Example: the first row uses 0.1*CS + 0.1*win_rate - 0.3*vision.
# A zero weight means that stat does not affect that output.

            #CS/min win_rate vision
weights = [ [0.1, 0.1, -0.3], #damage_taken?
            [0.1, 0.2, 0.0], #win_chance?
            [0.0, 1.3, 0.1] ] #tilt_level?

def w_sum(a,b):
    assert(len(a) == len(b))
    output = 0
    for i in range(len(a)):
        output += (a[i] * b[i])
    return output

def vect_mat_mul(vect,matrix):
    assert(len(vect) == len(matrix))
    output = [0,0,0]
    for i in range(len(vect)):
        output[i] = w_sum(vect,matrix[i])
    return output

def neural_network(input, weights):
    pred = vect_mat_mul(input,weights)
    return pred

# This dataset is the current
# status at the beginning of
# each game for four League of Legends matches.

# cs_per_min = CS per minute
# win_rate = recent win rate
# vision_score = vision score (scaled)

cs_per_min =  [8.5, 9.5, 9.9, 9.0]
win_rate = [0.65,0.8, 0.8, 0.9]
vision_score = [1.2, 1.3, 0.5, 1.0]

# Input corresponds to every entry
# for the first game of the season.

input = [cs_per_min[0],win_rate[0],vision_score[0]]
pred = neural_network(input,weights)

print(pred)

[0.555, 0.9800000000000001, 0.9650000000000001]


# Predicting on Predictions

In [6]:
# LoL example: the first small 'deep' network.
# Hidden layer = three in-between scores made from the match stats.
# It first mixes CS, win rate and vision into hid[0], hid[1], hid[2].
# Then it mixes those hidden scores to predict the three outputs.
# The hidden values are not extra data we typed in; the network makes them.

            #CS/min win_rate vision
ih_wgt = [ [0.1, 0.2, -0.1], #hid[0]
           [-0.1,0.1, 0.9], #hid[1]
           [0.1, 0.4, 0.1] ] #hid[2]

           #hid[0] hid[1] hid[2]
hp_wgt = [ [0.3, 1.1, -0.3], #damage_taken?
           [0.1, 0.2, 0.0], #win_chance?
           [0.0, 1.3, 0.1] ] #tilt_level?

weights = [ih_wgt, hp_wgt]

def neural_network(input, weights):
    hid = vect_mat_mul(input,weights[0])
    pred = vect_mat_mul(hid,weights[1])
    return pred

cs_per_min =  [8.5, 9.5, 9.9, 9.0]
win_rate = [0.65,0.8, 0.8, 0.9]
vision_score = [1.2, 1.3, 0.5, 1.0]

# Input corresponds to every entry
# for the first game of the season.

input = [cs_per_min[0],win_rate[0],vision_score[0]]
pred = neural_network(input,weights)

print(pred)

[0.21350000000000002, 0.14500000000000002, 0.5065]


# NumPy Version

In [7]:
# Same two-layer LoL prediction using NumPy.
# .T turns each 3x3 list of output rows into the orientation used by dot.
# The match input (3,) dot first weights (3,3) gives hidden scores (3,).
# Hidden scores (3,) dot second weights (3,3) gives outputs (3,).

import numpy as np

#CS/min win_rate vision
ih_wgt = np.array([ 
            [0.1, 0.2, -0.1], #hid[0]
            [-0.1,0.1, 0.9], #hid[1]
            [0.1, 0.4, 0.1]]).T #hid[2]


# hid[0] hid[1] hid[2]
hp_wgt = np.array([  
            [0.3, 1.1, -0.3], #damage_taken?
            [0.1, 0.2, 0.0], #win_chance?
            [0.0, 1.3, 0.1] ]).T #tilt_level?

weights = [ih_wgt, hp_wgt]

def neural_network(input, weights):

    hid = input.dot(weights[0])
    pred = hid.dot(weights[1])
    return pred


cs_per_min =  np.array([8.5, 9.5, 9.9, 9.0])
win_rate = np.array([0.65,0.8, 0.8, 0.9])
vision_score = np.array([1.2, 1.3, 0.5, 1.0])

input = np.array([cs_per_min[0],win_rate[0],vision_score[0]])

pred = neural_network(input,weights)
print(pred)

[0.2135 0.145  0.5065]


# A Quick Primer on NumPy

In [8]:
# NumPy warm-up: think of vectors as a champion's stats.
# A vector is one line of stats. A matrix has many lines.
# A shape tells us how many rows and columns we have.

import numpy as np

a = np.array([0,1,2,3]) # a vector
b = np.array([4,5,6,7]) # another vector
c = np.array([[0,1,2,3], # a matrix
              [4,5,6,7]])

d = np.zeros((2,4)) # (2x4 matrix of zeros)
e = np.random.rand(2,5) # random 2x5
# matrix with all numbers between 0 and 1

print(a)
print(b)
print(c)
print(d)
print(e)

[0 1 2 3]
[4 5 6 7]
[[0 1 2 3]
 [4 5 6 7]]
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]]
[[0.15199254 0.87141222 0.78460967 0.86775713 0.15049185]
 [0.64486012 0.04701434 0.04960525 0.86971818 0.84627032]]


In [9]:
# * multiplies values item by item; it is NOT the same as .dot.
# The last example (a * e) is meant to fail: 4 values cannot
# line up with the 5 columns in e. This is a shape mismatch.

print(a * 0.1) # multiplies every number in vector "a" by 0.1
      
print(c * 0.2) # multiplies every number in matrix "c" by 0.2
      
print(a * b) # multiplies elementwise between a and b (columns paired up)
      
print(a * b * 0.2) # elementwise multiplication then multiplied by 0.2
      
print(a * c) # since c has the same number of columns as a, this performs
# elementwise multiplication on every row of the matrix "c"

print(a * e) # since a and e don't have the same number of columns, this
# throws a "Value Error: operands could not be broadcast together with.."

[0.  0.1 0.2 0.3]
[[0.  0.2 0.4 0.6]
 [0.8 1.  1.2 1.4]]
[ 0  5 12 21]
[0.  1.  2.4 4.2]
[[ 0  1  4  9]
 [ 0  5 12 21]]


ValueError: operands could not be broadcast together with shapes (4,) (2,5) 

In [10]:
# LoL shape rule for .dot: inner sizes must match.
# (1,4) dot (4,3) works because 4 = 4. Output shape: (1,3).
# Imagine 4 stats being turned into 3 predictions.

a = np.zeros((1,4)) # vector of length 4
b = np.zeros((4,3)) # matrix with 4 rows & 3 columns

c = a.dot(b)
print(c.shape)

(1, 3)


In [11]:
# More LoL matrix practice: (2,4) dot (4,3) becomes (2,3).
# The INSIDE numbers must match; the OUTSIDE numbers are the result.
# (2,1) dot (1,3) -> (2,3); (4,5) dot (5,6) -> (4,6).
# The last example fails: (5,4) dot (5,6) has 4 != 5.
# That error is intentional so we can learn the shape rule.

a = np.zeros((2,4)) # matrix with 2 rows and 4 columns
b = np.zeros((4,3)) # matrix with 4 rows & 3 columns

c = a.dot(b)
print(c.shape) # outputs (2,3)

e = np.zeros((2,1)) # matrix with 2 rows and 1 columns
f = np.zeros((1,3)) # matrix with 1 row & 3 columns

g = e.dot(f)
print(g.shape) # outputs (2,3)

h = np.zeros((5,4)).T # matrix with 4 rows and 5 columns
i = np.zeros((5,6)) # matrix with 5 rows & 6 columns

j = h.dot(i)
print(j.shape) # outputs (4,6)

h = np.zeros((5,4)) # matrix with 5 rows and 4 columns
i = np.zeros((5,6)) # matrix with 5 rows & 6 columns
j = h.dot(i)
print(j.shape) # throws an error

(2, 3)
(2, 3)
(4, 6)


ValueError: shapes (5,4) and (5,6) not aligned: 4 (dim 1) != 5 (dim 0)